In [4]:
import pandas as pd
import gzip
import os
import re
from tqdm.notebook import tqdm  # For displaying progress bar in Jupyter Notebook
#
from natsort import natsorted
###

In [5]:
# Function to extract chromosome and position from the filename
def extract_chr_pos(file_name):
    match = re.search(r'(\d+)_([0-9]+)-', file_name)
    if match:
        chromosome = int(match.group(1))  # Extracted chromosome as integer
        position = int(match.group(2))    # Extracted start position as integer
        return chromosome, position
    return None, None

# Function to process each gzip file
def process_gzip_file(file_path):
    with gzip.open(file_path, 'rt') as f:
        df = pd.read_csv(f, sep='\t')

    # Step 1: Create 'SNPID' with 'chr' prefix
    df['SNPID'] = 'chr' + df['Chr'].astype(str) + ':' + df['Pos'].astype(str) + ':' + df['Ref'] + ':' + df['Alt']
    
    # Step 2: Create 'N' as sum of 'Num_Cases' and 'Num_Controls'
    df['N'] = df['Num_Cases'] + df['Num_Controls']
    
    # Step 3: Extract BETA, SE, and INFO from 'Info' column
    df['BETA'] = df['Info'].str.extract(r'REGENIE_BETA=([-+]?\d*\.\d+|\d+)')
    df['SE'] = df['Info'].str.extract(r'REGENIE_SE=([-+]?\d*\.\d+|\d+)')
    df['INFO'] = df['Info'].str.extract(r'INFO=([-+]?\d*\.\d+|\d+)')
    
    # Step 4: Select the required columns
    df_output = df[['SNPID', 'Ref', 'Alt', 'AAF', 'BETA', 'SE', 'Pval', 'Num_Cases', 'N', 'INFO']]
    
    # Rename columns for final output
    df_output.columns = ['SNPID', 'REF', 'ALT', 'AAF', 'BETA', 'SE', 'P', 'Num_Cases', 'N', 'INFO']
    
    return df_output

In [23]:
# Main processing function for each trait
def process_trait_files(trait_name, file_list, output_dir, output_prefix):
    combined_df = pd.DataFrame()

    # Sort files by chromosome and position
    # sorted_files = sorted(file_list, key=lambda x: extract_chr_pos(x))
    
    # Sort -V type function in python
    sorted_files = natsorted(file_list)

    # Progress bar for file processing 
    for file in tqdm(sorted_files, desc=f'Processing files for trait {trait_name}'):
        
        print(os.path.basename(file))
        
        df = process_gzip_file(file)
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    
    # Sort combined dataframe by SNPID
    # combined_df = combined_df.sort_values(by=['SNPID'])

    # Create output file path with directory and prefix
    output_file = os.path.join(output_dir, f'{output_prefix}_combined_{trait_name}.tsv')

    # Save the final concatenated and sorted dataframe to a new file
    combined_df.to_csv(output_file, sep='\t', index=False)


In [24]:
# Folder containing the gzip files
folder_path = '/broad/hptmp/mesbah/dataset/ch_gwas/ukbb/MultiANC/ukb200k_N193342/'

# Output directory and file prefix
output_dir = '/broad/hptmp/mesbah/dataset/ch_gwas/ukbb/MultiANC/combined/'  # Directory to store output files

output_prefix = 'chr1_22.ukb200k_N193342'     # Prefix for output file names

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# List of traits to process
traits = ['CH', 'CHvaf05', 'CHvaf10', 'DNMT3A', 'TET2', 'ASXL1', 'SF', 'DDR']

# traits = ['ASXL1', 'SF', 'DDR']

# Progress bar for trait processing
for trait in tqdm(traits, desc='Processing all traits'):
    # Filter out the files for the current trait
    trait_files = [os.path.join(folder_path, file) for file in os.listdir(folder_path) if trait in file and file.endswith(f'has{trait}.regenie.gz')]  
    # Process all files for this trait and create a single combined file
    process_trait_files(trait, trait_files, output_dir, output_prefix)


Processing all traits:   0%|          | 0/8 [00:00<?, ?it/s]

Processing files for trait CH:   0%|          | 0/159 [00:00<?, ?it/s]

chr1_MultiAnc.ukb200k_N193342.1_1-24895642_hasCH.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_24895643-49791284_hasCH.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_49791285-74686926_hasCH.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_74686927-99582568_hasCH.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_99582569-124478211_hasCH.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_124478212-149373853_hasCH.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_149373854-174269495_hasCH.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_174269496-199165137_hasCH.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_199165138-224060779_hasCH.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_224060780-248956422_hasCH.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_1-24219352_hasCH.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_24219353-48438705_hasCH.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_48438706-72658058_hasCH.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_72658059-96877411_hasCH.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_96877412-121096764_hasCH.regenie.

chr15_MultiAnc.ukb200k_N193342.15_40796476-61194713_hasCH.regenie.gz
chr15_MultiAnc.ukb200k_N193342.15_61194714-81592951_hasCH.regenie.gz
chr15_MultiAnc.ukb200k_N193342.15_81592952-101991189_hasCH.regenie.gz
chr16_MultiAnc.ukb200k_N193342.16_1-18067669_hasCH.regenie.gz
chr16_MultiAnc.ukb200k_N193342.16_18067670-36135338_hasCH.regenie.gz
chr16_MultiAnc.ukb200k_N193342.16_36135339-54203007_hasCH.regenie.gz
chr16_MultiAnc.ukb200k_N193342.16_54203008-72270676_hasCH.regenie.gz
chr16_MultiAnc.ukb200k_N193342.16_72270677-90338345_hasCH.regenie.gz
chr17_MultiAnc.ukb200k_N193342.17_1-16651488_hasCH.regenie.gz
chr17_MultiAnc.ukb200k_N193342.17_16651489-33302976_hasCH.regenie.gz
chr17_MultiAnc.ukb200k_N193342.17_33302977-49954464_hasCH.regenie.gz
chr17_MultiAnc.ukb200k_N193342.17_49954465-66605952_hasCH.regenie.gz
chr17_MultiAnc.ukb200k_N193342.17_66605953-83257441_hasCH.regenie.gz
chr18_MultiAnc.ukb200k_N193342.18_1-16074657_hasCH.regenie.gz
chr18_MultiAnc.ukb200k_N193342.18_16074658-32149314_ha

Processing files for trait CHvaf05:   0%|          | 0/159 [00:00<?, ?it/s]

...

Processing files for trait TET2:   0%|          | 0/159 [00:00<?, ?it/s]

chr1_MultiAnc.ukb200k_N193342.1_1-24895642_hasTET2.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_24895643-49791284_hasTET2.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_49791285-74686926_hasTET2.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_74686927-99582568_hasTET2.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_99582569-124478211_hasTET2.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_124478212-149373853_hasTET2.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_149373854-174269495_hasTET2.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_174269496-199165137_hasTET2.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_199165138-224060779_hasTET2.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_224060780-248956422_hasTET2.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_1-24219352_hasTET2.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_24219353-48438705_hasTET2.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_48438706-72658058_hasTET2.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_72658059-96877411_hasTET2.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_96877

chr14_MultiAnc.ukb200k_N193342.14_85634975-107043718_hasTET2.regenie.gz
chr15_MultiAnc.ukb200k_N193342.15_1-20398237_hasTET2.regenie.gz
chr15_MultiAnc.ukb200k_N193342.15_20398238-40796475_hasTET2.regenie.gz
chr15_MultiAnc.ukb200k_N193342.15_40796476-61194713_hasTET2.regenie.gz
chr15_MultiAnc.ukb200k_N193342.15_61194714-81592951_hasTET2.regenie.gz
chr15_MultiAnc.ukb200k_N193342.15_81592952-101991189_hasTET2.regenie.gz
chr16_MultiAnc.ukb200k_N193342.16_1-18067669_hasTET2.regenie.gz
chr16_MultiAnc.ukb200k_N193342.16_18067670-36135338_hasTET2.regenie.gz
chr16_MultiAnc.ukb200k_N193342.16_36135339-54203007_hasTET2.regenie.gz
chr16_MultiAnc.ukb200k_N193342.16_54203008-72270676_hasTET2.regenie.gz
chr16_MultiAnc.ukb200k_N193342.16_72270677-90338345_hasTET2.regenie.gz
chr17_MultiAnc.ukb200k_N193342.17_1-16651488_hasTET2.regenie.gz
chr17_MultiAnc.ukb200k_N193342.17_16651489-33302976_hasTET2.regenie.gz
chr17_MultiAnc.ukb200k_N193342.17_33302977-49954464_hasTET2.regenie.gz
chr17_MultiAnc.ukb200k_N1

Processing files for trait ASXL1:   0%|          | 0/159 [00:00<?, ?it/s]

chr1_MultiAnc.ukb200k_N193342.1_1-24895642_hasASXL1.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_24895643-49791284_hasASXL1.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_49791285-74686926_hasASXL1.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_74686927-99582568_hasASXL1.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_99582569-124478211_hasASXL1.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_124478212-149373853_hasASXL1.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_149373854-174269495_hasASXL1.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_174269496-199165137_hasASXL1.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_199165138-224060779_hasASXL1.regenie.gz
chr1_MultiAnc.ukb200k_N193342.1_224060780-248956422_hasASXL1.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_1-24219352_hasASXL1.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_24219353-48438705_hasASXL1.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_48438706-72658058_hasASXL1.regenie.gz
chr2_MultiAnc.ukb200k_N193342.2_72658059-96877411_hasASXL1.regenie.gz
chr2_MultiAnc.ukb200k_N